In [6]:
# ============================================================
# Milestone 7.2 / 7.3 — Dataset & Environment Sanity Check
# TEMPORARY NOTEBOOK CELL — DELETE AFTER DEBUGGING
# ============================================================

from pathlib import Path
import sys
import os
import pandas as pd


# ============================================================
# 1. Locate project root
# ============================================================

cwd = Path.cwd().resolve()

print("=" * 70)
print("ENVIRONMENT CHECK")
print("=" * 70)

print("Current working directory:")
print(cwd)


project_root = None

for candidate in [cwd, *cwd.parents]:

    if (
        (candidate / "src").is_dir()
        and
        (candidate / "results").is_dir()
    ):
        project_root = candidate
        break


if project_root is None:

    raise RuntimeError(
        "Could not locate thesis project root.\n"
        "Expected a directory containing both 'src/' and 'results/'."
    )


print("\nProject root:")
print(project_root)


# ============================================================
# 2. Add project root to Python path
# ============================================================

project_root_str = str(project_root)

if project_root_str not in sys.path:

    sys.path.insert(
        0,
        project_root_str,
    )


print("\nPython path:")
print(sys.path[0])


# ============================================================
# 3. Import project config
# ============================================================

from src.baseline import config


print("\nConfig imported successfully.")


# ============================================================
# 4. Locate six-label dataset
# ============================================================

DATASET_PATH = (
    project_root
    / "results"
    / "six_label_patient_level_dataset"
    / "labeled_dataset.csv"
)


print("\nDataset path:")
print(DATASET_PATH)


if not DATASET_PATH.exists():

    raise FileNotFoundError(
        f"Six-label dataset not found:\n{DATASET_PATH}"
    )


# ============================================================
# 5. Load dataset
# ============================================================

df = pd.read_csv(
    DATASET_PATH,
    nrows=5,
)


print("\nDataset loaded successfully.")

print(
    "Number of columns:",
    len(df.columns),
)


# ============================================================
# 6. Show label configuration
# ============================================================

print("\n" + "=" * 70)
print("CONFIG LABELS")
print("=" * 70)

print(
    "config.NUM_LABELS:",
    config.NUM_LABELS,
)

print(
    "config.LABEL_NAMES:"
)

for i, label in enumerate(
    config.LABEL_NAMES,
    start=1,
):

    print(
        f"  {i}. {label}"
    )


# ============================================================
# 7. Detect label columns in dataset
# ============================================================

dataset_label_columns = [
    column
    for column in df.columns
    if column.startswith("label_")
]


print("\n" + "=" * 70)
print("DATASET LABEL COLUMNS")
print("=" * 70)

print(
    "Number of label columns:",
    len(dataset_label_columns),
)

for i, label in enumerate(
    dataset_label_columns,
    start=1,
):

    print(
        f"  {i}. {label}"
    )


# ============================================================
# 8. Compare config vs dataset
# ============================================================

config_labels = set(
    config.LABEL_NAMES
)

dataset_labels = set(
    dataset_label_columns
)

missing_from_dataset = sorted(
    config_labels - dataset_labels
)

extra_in_dataset = sorted(
    dataset_labels - config_labels
)


print("\n" + "=" * 70)
print("LABEL COMPATIBILITY")
print("=" * 70)

if not missing_from_dataset:

    print(
        "✓ All config labels exist in dataset."
    )

else:

    print(
        "❌ Labels expected by config but missing "
        "from dataset:"
    )

    for label in missing_from_dataset:

        print(
            f"   - {label}"
        )


if not extra_in_dataset:

    print(
        "✓ Dataset has no unexpected label columns."
    )

else:

    print(
        "⚠ Dataset contains additional label columns:"
    )

    for label in extra_in_dataset:

        print(
            f"   - {label}"
        )


# ============================================================
# 9. Show relevant dataset columns
# ============================================================

print("\n" + "=" * 70)
print("RELEVANT DATASET COLUMNS")
print("=" * 70)

relevant_columns = [
    column
    for column in [
        "split",
        "checkup_id",
        "patient_id",
        "photographs",
        "radiographs",
        "chief_complaint",
        "present_illness",
        "past_medical_record",
        "examination",
    ]
    if column in df.columns
]

relevant_columns += dataset_label_columns

for column in relevant_columns:

    print(
        f"  ✓ {column}"
    )


# ============================================================
# 10. Final verdict
# ============================================================

print("\n" + "=" * 70)

if (
    len(dataset_label_columns) == config.NUM_LABELS
    and
    not missing_from_dataset
):

    print(
        "✓ ENVIRONMENT + DATASET CHECK: PASS"
    )

else:

    print(
        "❌ ENVIRONMENT + DATASET CHECK: FAIL"
    )

    print(
        "\nDo NOT continue to representation extraction "
        "or fusion training yet."
    )

print("=" * 70)

ENVIRONMENT CHECK
Current working directory:
/home/ubuntu/Projects/thesis-code/notebooks

Project root:
/home/ubuntu/Projects/thesis-code

Python path:
/home/ubuntu/Projects/thesis-code

Config imported successfully.

Dataset path:
/home/ubuntu/Projects/thesis-code/results/six_label_patient_level_dataset/labeled_dataset.csv

Dataset loaded successfully.
Number of columns: 46

CONFIG LABELS
config.NUM_LABELS: 6
config.LABEL_NAMES:
  1. label_caries
  2. label_gingivitis
  3. label_malocclusion
  4. label_pulpitis
  5. label_tooth_loss
  6. label_tooth_structure_loss

DATASET LABEL COLUMNS
Number of label columns: 6
  1. label_gingivitis
  2. label_tooth_structure_loss
  3. label_pulpitis
  4. label_tooth_loss
  5. label_caries
  6. label_malocclusion

LABEL COMPATIBILITY
✓ All config labels exist in dataset.
✓ Dataset has no unexpected label columns.

RELEVANT DATASET COLUMNS
  ✓ split
  ✓ checkup_id
  ✓ patient_id
  ✓ photographs
  ✓ radiographs
  ✓ chief_complaint
  ✓ present_illness


In [7]:
# ============================================================
# Milestone 7.3 — Fusion Representation Dataset Test
# TEMPORARY NOTEBOOK CELL — DELETE AFTER DEBUGGING
# ============================================================

from pathlib import Path
import sys
import torch


# ============================================================
# 1. Locate project root
# ============================================================

cwd = Path.cwd().resolve()

project_root = None

for candidate in [cwd, *cwd.parents]:

    if (
        (candidate / "src").is_dir()
        and
        (candidate / "results").is_dir()
    ):
        project_root = candidate
        break


if project_root is None:
    raise RuntimeError(
        "Could not locate thesis project root."
    )


project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)


# ============================================================
# 2. Import Dataset
# ============================================================

from src.fusion.dataset import (
    FusionRepresentationDataset,
)


# ============================================================
# 3. Configuration
# ============================================================

REPRESENTATION_ROOT = (
    project_root
    / "results"
    / "fusion"
    / "ssl_representations"
)


# ============================================================
# 4. Test each split
# ============================================================

print("=" * 70)
print("MILESTONE 7.3 — FUSION DATASET CHECK")
print("=" * 70)

datasets = {}

for split in [
    "train",
    "validation",
    "test",
]:

    print()
    print("-" * 70)
    print(f"Testing split: {split}")
    print("-" * 70)

    dataset = FusionRepresentationDataset(
        REPRESENTATION_ROOT,
        split,
    )

    datasets[split] = dataset

    print(
        "Length:",
        len(dataset),
    )

    # --------------------------------------------------------
    # First sample
    # --------------------------------------------------------

    sample = dataset[0]

    print("\nFirst sample:")

    for key, value in sample.items():

        if torch.is_tensor(value):

            print(
                f"  {key:12s}: "
                f"shape={tuple(value.shape)}, "
                f"dtype={value.dtype}"
            )

        else:

            print(
                f"  {key:12s}: "
                f"type={type(value).__name__}, "
                f"value={value}"
            )

    # --------------------------------------------------------
    # Shape checks
    # --------------------------------------------------------

    assert sample["image"].shape == (
        2048,
    )

    assert sample["radiograph"].shape == (
        2048,
    )

    assert sample["text"].shape == (
        768,
    )

    assert sample["labels"].shape == (
        6,
    )

    # --------------------------------------------------------
    # Numerical checks
    # --------------------------------------------------------

    assert torch.isfinite(
        sample["image"]
    ).all()

    assert torch.isfinite(
        sample["radiograph"]
    ).all()

    assert torch.isfinite(
        sample["text"]
    ).all()

    assert torch.isfinite(
        sample["labels"]
    ).all()

    # --------------------------------------------------------
    # Label checks
    # --------------------------------------------------------

    unique_labels = torch.unique(
        sample["labels"]
    ).tolist()

    assert all(
        value in [0.0, 1.0]
        for value in unique_labels
    )

    print("\n✓ Sample shape check: PASS")
    print("✓ Numerical validity: PASS")
    print("✓ Binary label check: PASS")


# ============================================================
# 5. DataLoader test
# ============================================================

print()
print("=" * 70)
print("DATALOADER CHECK")
print("=" * 70)

from torch.utils.data import DataLoader


train_loader = DataLoader(
    datasets["train"],
    batch_size=4,
    shuffle=False,
    num_workers=0,
)


batch = next(
    iter(train_loader)
)


print("\nBatch keys:")

for key in batch:
    print(
        f"  ✓ {key}"
    )


print("\nBatch shapes:")

for key, value in batch.items():

    if torch.is_tensor(value):

        print(
            f"  {key:12s}: "
            f"{tuple(value.shape)}"
        )

    else:

        print(
            f"  {key:12s}: "
            f"type={type(value).__name__}"
        )


# ============================================================
# 6. Expected batch shapes
# ============================================================

assert batch["image"].shape == (
    4,
    2048,
)

assert batch["radiograph"].shape == (
    4,
    2048,
)

assert batch["text"].shape == (
    4,
    768,
)

assert batch["labels"].shape == (
    4,
    6,
)


# ============================================================
# 7. Final verdict
# ============================================================

print()
print("=" * 70)
print("MILESTONE 7.3 — DATASET CHECK: PASS")
print("=" * 70)

print(
    "\nAll three splits load correctly."
)

print(
    "FusionRepresentationDataset output is "
    "compatible with the fusion trainer."
)

print(
    "\nNext step:"
)

print(
    "→ Test SimpleFusion / MainFusion forward pass"
)

print("=" * 70)

MILESTONE 7.3 — FUSION DATASET CHECK

----------------------------------------------------------------------
Testing split: train
----------------------------------------------------------------------
train: 2935 SSL representation samples
Length: 2935

First sample:
  checkup_id  : type=str, value=0001-001
  patient_id  : type=str, value=1
  image       : shape=(2048,), dtype=torch.float32
  radiograph  : shape=(2048,), dtype=torch.float32
  text        : shape=(768,), dtype=torch.float32
  labels      : shape=(6,), dtype=torch.float32

✓ Sample shape check: PASS
✓ Numerical validity: PASS
✓ Binary label check: PASS

----------------------------------------------------------------------
Testing split: validation
----------------------------------------------------------------------
validation: 627 SSL representation samples
Length: 627

First sample:
  checkup_id  : type=str, value=0035-001
  patient_id  : type=str, value=35
  image       : shape=(2048,), dtype=torch.float32
  radiogr

In [8]:
# ============================================================
# Milestone 7.3 — Fusion Model Forward-Pass Test
# TEMPORARY NOTEBOOK CELL — DELETE AFTER DEBUGGING
# ============================================================

from pathlib import Path
import sys
import torch


# ============================================================
# 1. Locate project root
# ============================================================

cwd = Path.cwd().resolve()

project_root = None

for candidate in [cwd, *cwd.parents]:

    if (
        (candidate / "src").is_dir()
        and
        (candidate / "results").is_dir()
    ):
        project_root = candidate
        break


if project_root is None:
    raise RuntimeError(
        "Could not locate thesis project root."
    )


project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)


# ============================================================
# 2. Imports
# ============================================================

from src.fusion.dataset import (
    FusionRepresentationDataset,
)

from src.fusion.fusion_model import (
    SimpleFusion,
    MainFusion,
)

from src.baseline import config


# ============================================================
# 3. Configuration
# ============================================================

REPRESENTATION_ROOT = (
    project_root
    / "results"
    / "fusion"
    / "ssl_representations"
)

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

BATCH_SIZE = 4


# ============================================================
# 4. Header
# ============================================================

print("=" * 70)
print("MILESTONE 7.3 — FUSION MODEL FORWARD-PASS CHECK")
print("=" * 70)

print(
    "Project root:",
    project_root,
)

print(
    "Device:",
    DEVICE,
)

print(
    "Number of labels:",
    config.NUM_LABELS,
)


# ============================================================
# 5. Load validation representation dataset
# ============================================================

dataset = FusionRepresentationDataset(
    REPRESENTATION_ROOT,
    "validation",
)


# ============================================================
# 6. Create a small batch
# ============================================================

from torch.utils.data import DataLoader


loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)


batch = next(
    iter(loader)
)


image = batch["image"].to(
    DEVICE
)

radiograph = batch["radiograph"].to(
    DEVICE
)

text = batch["text"].to(
    DEVICE
)

labels = batch["labels"].to(
    DEVICE
)


print()
print("=" * 70)
print("INPUT BATCH")
print("=" * 70)

print(
    "image:",
    tuple(image.shape),
)

print(
    "radiograph:",
    tuple(radiograph.shape),
)

print(
    "text:",
    tuple(text.shape),
)

print(
    "labels:",
    tuple(labels.shape),
)


# ============================================================
# 7. Input validation
# ============================================================

assert image.shape == (
    BATCH_SIZE,
    2048,
)

assert radiograph.shape == (
    BATCH_SIZE,
    2048,
)

assert text.shape == (
    BATCH_SIZE,
    768,
)

assert labels.shape == (
    BATCH_SIZE,
    config.NUM_LABELS,
)


assert torch.isfinite(
    image
).all()

assert torch.isfinite(
    radiograph
).all()

assert torch.isfinite(
    text
).all()

assert torch.isfinite(
    labels
).all()


print()
print("✓ Input shapes: PASS")
print("✓ Input numerical validity: PASS")


# ============================================================
# 8. Test SimpleFusion
# ============================================================

print()
print("=" * 70)
print("TESTING SimpleFusion")
print("=" * 70)


simple_model = SimpleFusion(
    image_dim=2048,
    radiograph_dim=2048,
    text_dim=768,
    hidden_dim=512,
    num_labels=config.NUM_LABELS,
    dropout=0.3,
).to(DEVICE)


simple_model.eval()


with torch.no_grad():

    simple_logits = simple_model(
        image,
        radiograph,
        text,
    )


print(
    "Output shape:",
    tuple(simple_logits.shape),
)

print(
    "Output dtype:",
    simple_logits.dtype,
)

print(
    "Output finite:",
    torch.isfinite(
        simple_logits
    ).all().item(),
)


assert simple_logits.shape == (
    BATCH_SIZE,
    config.NUM_LABELS,
)

assert torch.isfinite(
    simple_logits
).all()


print("✓ SimpleFusion forward pass: PASS")


# ============================================================
# 9. Test MainFusion
# ============================================================

print()
print("=" * 70)
print("TESTING MainFusion")
print("=" * 70)


main_model = MainFusion(
    image_dim=2048,
    radiograph_dim=2048,
    text_dim=768,
    modality_dim=512,
    hidden_dim=512,
    num_labels=config.NUM_LABELS,
    dropout=0.3,
).to(DEVICE)


main_model.eval()


with torch.no_grad():

    main_logits = main_model(
        image,
        radiograph,
        text,
    )


print(
    "Output shape:",
    tuple(main_logits.shape),
)

print(
    "Output dtype:",
    main_logits.dtype,
)

print(
    "Output finite:",
    torch.isfinite(
        main_logits
    ).all().item(),
)


assert main_logits.shape == (
    BATCH_SIZE,
    config.NUM_LABELS,
)

assert torch.isfinite(
    main_logits
).all()


print("✓ MainFusion forward pass: PASS")


# ============================================================
# 10. Test loss compatibility
# ============================================================

print()
print("=" * 70)
print("LOSS COMPATIBILITY CHECK")
print("=" * 70)


import torch.nn as nn


criterion = nn.BCEWithLogitsLoss()


simple_loss = criterion(
    simple_logits,
    labels,
)

main_loss = criterion(
    main_logits,
    labels,
)


print(
    f"SimpleFusion BCE loss: "
    f"{simple_loss.item():.6f}"
)

print(
    f"MainFusion BCE loss:   "
    f"{main_loss.item():.6f}"
)


assert torch.isfinite(
    simple_loss
)

assert torch.isfinite(
    main_loss
)


print("✓ BCEWithLogitsLoss compatibility: PASS")


# ============================================================
# 11. Parameter counts
# ============================================================

print()
print("=" * 70)
print("MODEL SIZE")
print("=" * 70)


simple_params = sum(
    parameter.numel()
    for parameter in simple_model.parameters()
)


main_params = sum(
    parameter.numel()
    for parameter in main_model.parameters()
)


print(
    f"SimpleFusion parameters: "
    f"{simple_params:,}"
)

print(
    f"MainFusion parameters:   "
    f"{main_params:,}"
)


# ============================================================
# 12. Final verdict
# ============================================================

print()
print("=" * 70)
print("MILESTONE 7.3 — FORWARD-PASS CHECK: PASS")
print("=" * 70)

print()
print("✓ Representation dataset loads correctly")
print("✓ SimpleFusion accepts the representations")
print("✓ MainFusion accepts the representations")
print("✓ Both models produce 6-label logits")
print("✓ BCEWithLogitsLoss is compatible")
print()
print("Next step:")
print("→ Run actual Milestone 7.3 fusion training")
print("=" * 70)

MILESTONE 7.3 — FUSION MODEL FORWARD-PASS CHECK
Project root: /home/ubuntu/Projects/thesis-code
Device: cuda
Number of labels: 6
validation: 627 SSL representation samples

INPUT BATCH
image: (4, 2048)
radiograph: (4, 2048)
text: (4, 768)
labels: (4, 6)

✓ Input shapes: PASS
✓ Input numerical validity: PASS

TESTING SimpleFusion
Output shape: (4, 6)
Output dtype: torch.float32
Output finite: True
✓ SimpleFusion forward pass: PASS

TESTING MainFusion
Output shape: (4, 6)
Output dtype: torch.float32
Output finite: True
✓ MainFusion forward pass: PASS

LOSS COMPATIBILITY CHECK
SimpleFusion BCE loss: 0.684202
MainFusion BCE loss:   0.690399
✓ BCEWithLogitsLoss compatibility: PASS

MODEL SIZE
SimpleFusion parameters: 2,493,958
MainFusion parameters:   3,281,926

MILESTONE 7.3 — FORWARD-PASS CHECK: PASS

✓ Representation dataset loads correctly
✓ SimpleFusion accepts the representations
✓ MainFusion accepts the representations
✓ Both models produce 6-label logits
✓ BCEWithLogitsLoss is compa

In [9]:
# ============================================================
# TEMPORARY CHECK — Fusion model filename
# DELETE AFTER DEBUGGING
# ============================================================

from pathlib import Path

project_root = Path("/home/ubuntu/Projects/thesis-code")

fusion_dir = project_root / "src" / "fusion"

print("=" * 70)
print("FUSION DIRECTORY CHECK")
print("=" * 70)

print("\nFusion directory:")
print(fusion_dir)

print("\nFiles:")

for path in sorted(fusion_dir.iterdir()):
    print(f"  {path.name}")

print("\n" + "=" * 70)

print("model.py exists:",
      (fusion_dir / "model.py").exists())

print("fusion_model.py exists:",
      (fusion_dir / "fusion_model.py").exists())

print("=" * 70)

FUSION DIRECTORY CHECK

Fusion directory:
/home/ubuntu/Projects/thesis-code/src/fusion

Files:
  __init__.py
  __pycache__
  dataset.py
  evaluate_fusion.py
  fusion_model.py
  representation_extractor.py
  representations.py
  test_protocol.py
  train_fusion.py

model.py exists: False
fusion_model.py exists: True
